# PS-S5E12: Model Stacking

This notebook tackles the [**Playground Series – Season 6, Episode 1: Diabetes Prediction Challenge**](https://www.kaggle.com/competitions/playground-series-s6e1), a competition focused on predicting the exam score for a student, based on features describing the student, the exam, and the facility.

## Ensemble Strategy: Stacking with Ridge Regression

While simple **Blending** (Weighted Averaging) assumes that the best prediction is just a sum of parts, **Stacking** (Stacked Generalization) treats the combination of models as a supervised learning problem in itself.

### The "Weak Model" Paradox
In a standard Hill Climbing or Weighted Average, a model with a poor score (e.g., Ridge Regression with RMSE 8.9) is often assigned a weight of 0 because it strictly worsens the score of a strong model (e.g., XGBoost with RMSE 8.6).
  
However, "weaker" models often capture patterns that strong models miss (e.g., linear trends vs. decision boundaries). We don't just want to average them; we want to use the linear model to correct the residuals of the tree model.
    
### The Solution: A Ridge Meta-Learner
Instead of guessing weights, we train a **Meta-Learner** (specifically `RidgeCV`) using the Out-of-Fold (OOF) predictions of our base models as *features*.
  1. **Input:** The input to this meta-model is not the student data, but the **predictions** made by XGBoost, CatBoost, Neural Net, etc.
  2. **Learning:** The meta-model learns conditional relationships. For example: *"When XGBoost predicts high but the Neural Network predicts low, the true answer tends to be in the middle."*
  3. **Bias Correction:** Unlike a simple blend, Ridge Regression allows for **negative coefficients**. This means the meta-learner can subtract a specific model's prediction to correct systematic bias, effectively using the weaker models as "error correctors" rather than primary predictors.

### Why RidgeCV?
We use **Ridge Regression** (Linear Regression with L2 Regularization) as the meta-learner because the predictions from our base models are highly correlated (Multicollinearity). Standard Linear Regression would fail here, but Ridge handles the correlation gracefully, finding the optimal stable mix that minimizes **RMSE**.

## Install Needed Packages

In [1]:
import os
import glob
import warnings

# --- Third-party
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import RidgeCV
import matplotlib.pyplot as plt

from ps_s06e01_experiment_setup import ExperimentSetup

# --- Notebook settings
warnings.filterwarnings('ignore')

%matplotlib inline

In [2]:
helper = ExperimentSetup()

# Get the seed, and apply it to all the internals like pandas and numpy
seed = helper.set_seeds()

helper.configure_pandas()
helper.suppress_warnings()

# The target feature
TARGET = 'exam_score'

Random seed set to: 10301
Warnings suppressed.


In [3]:
# Load Ground Truth
training_df = helper.read_training_dataset()
y_true = training_df[TARGET].values

TRAINING DATASET

   id  age  gender   course  study_hours  class_attendance internet_access  \
0   0   21  female     b.sc        7.910            98.800              no   
1   1   18   other  diploma        4.950            94.800             yes   
2   2   20  female     b.sc        4.680            92.600             yes   
3   3   19    male     b.sc        2.000            49.500             yes   
4   4   23    male      bca        7.650            86.900             yes   

   sleep_hours sleep_quality   study_method facility_rating exam_difficulty  \
0        4.900       average  online videos             low            easy   
1        4.700          poor     self-study          medium        moderate   
2        5.800          poor       coaching            high        moderate   
3        8.300       average    group study            high        moderate   
4        9.600          good     self-study            high            easy   

   exam_score  
0      78.300  
1     

In [4]:
submission_df = helper.read_sample_submission_dataset()

## Loading OOF and Test Predictions
We load the Out-of-Fold (OOF) predictions and Test predictions generated by our individual model notebooks.
*   **OOF Predictions:** used to train the ensemble weights. Because these predictions were made on data the models _didn't_ see during training, they provide an unbiased estimate of performance.
*   **Test Predictions:** The final target we want to predict. We will apply the learned weights to these.

In [5]:
oof_files = []
test_files = []

if helper.running_in_kaggle():
    pred_dir = '/kaggle/input/ps-s06e01-*/predictions'
else:
    pred_dir = 'predictions'

oof_files = sorted(glob.glob(f'{pred_dir}/*_oof_preds.csv'))
test_files = sorted(glob.glob(f'{pred_dir}/*_test_preds.csv'))

print(f"Found {len(oof_files)} OOF files and {len(test_files)} Test files.")

Found 5 OOF files and 5 Test files.


In [6]:
# Helper to load and merge predictions
def load_preds(file_list, index_col='id'):
    df_list = []
    for file in file_list:
        model_name = os.path.basename(file).replace('_oof_preds.csv', '').replace('_test_preds.csv', '')
        # Read file
        df = pd.read_csv(file)
        
        # If files contain 'id' and 'pred', index by id. 
        # Assuming the column naming convention is standard (e.g. 'pred_xgb', 'diagnosed_diabetes', or similar)
        # We will dynamically find the prediction column (not 'id' or 'target')
        pred_col = [c for c in df.columns if c not in ['id', TARGET]][0]
        
        df = df.rename(columns={pred_col: model_name})
        df = df.set_index(index_col)[model_name]
        df_list.append(df)
        
    return pd.concat(df_list, axis=1)

In [7]:
# Create DataFrames
oof_df = load_preds(oof_files)
test_df = load_preds(test_files)

# Make sure y_true is indexed by id and aligned to oof_df
train_y = training_df.set_index('id')[TARGET]

# Keep only rows where all models have predictions
oof_df = oof_df.sort_index()
oof_df = oof_df.dropna(axis=0)

# Align y_true to the remaining ids
y_true_aligned = train_y.loc[oof_df.index].values

print(f"OOF Shape: {oof_df.shape}")
print(f"Test Shape: {test_df.shape}")

OOF Shape: (630000, 5)
Test Shape: (270000, 5)


In [8]:
# Check individual scores after transformation
print("\nIndividual Model RMSE (Ranked):")
best_single_model = None
best_single_score = 101

for model in oof_df.columns:
    score = np.sqrt(mean_squared_error(y_true_aligned, oof_df[model]))
    print(f"{model}: {score:.6f}")
    if score < best_single_score:
        best_single_score = score
        best_single_model = model

print(f"\nBest model/score: {best_single_model}/{best_single_score:.6f}")


Individual Model RMSE (Ranked):
cb: 8.709394
lgb: 8.707389
nn: 8.870686
ridge: 8.894346
xgb: 8.681841

Best model/score: xgb/8.681841


## Stacking Optimization: The Ridge Meta-Learner

Instead of using a greedy iterative algorithm (like Hill Climbing), we employ **Stacked Generalization** (Stacking). In this approach, we train a second-level "Meta-Learner" to optimally combine the predictions of our base models.
  
We use **Ridge Regression (L2 Regularized Linear Regression)** as our meta-learner.

1. **Construct the Level-2 Dataset:**
   * **Features ($X$):** The Out-of-Fold (OOF) predictions from our base models (XGBoost, LightGBM, CatBoost, Neural Net, Ridge).
   * **Target ($y$):** The actual exam scores from the training set.
2. **Train with Cross-Validation (`RidgeCV`):**
   * We fit the Ridge model to minimize **RMSE**.
   * The model automatically tunes its regularization parameter ($\alpha$) to handle **multicollinearity** (the fact that all our models predict very similar things).
3. **Learn Coefficients (Not Just Weights):**
   * Unlike simple blending, Ridge Regression learns **coefficients** that can be negative.
   * **Positive Coefficients:** Indicate the model is a primary signal provider (e.g., XGBoost).
   * **Negative Coefficients:** Indicate the model is useful for bias correction. If the Neural Network consistently over-predicts in a specific pattern where XGBoost under-predicts, the meta-learner subtracts the NN's prediction to "fix" the error.
4. **Final Prediction:**
   * The trained meta-model is applied to the Test set predictions to generate the final submission.

In [9]:
# ==========================================
# STACKING STRATEGY (Meta-Model)
# ==========================================

print("Preparing Stacking Data...")

# Get the correct column names
# We convert 'predictions/xgb_oof_preds.csv' -> 'xgb' to match oof_df columns
model_cols = [os.path.basename(f).replace('_oof_preds.csv', '') for f in oof_files]

print(f"Stacking Models: {model_cols}")

# Align Data
# X_stack = The OOF predictions from all models (features for the meta-learner)
X_stack = oof_df[model_cols].values
y_stack = y_true_aligned

# X_test_stack = The Test predictions from all models
# We map the OOF filenames to their corresponding Test filenames
X_test_stack = test_df[model_cols].values

# Train Meta-Learner (RidgeCV)
# RidgeCV automatically finds the best regularization (alpha) to prevent overfitting.
# We allow negative weights (unlike Hill Climbing) because sometimes subtracting 
# a biased model's prediction is useful.
meta_model = RidgeCV(
    alphas=[0.1, 1.0, 10.0, 50.0, 100.0, 500.0], 
    cv=5,
    scoring='neg_root_mean_squared_error'
)

print("Training Meta-Learner (Ridge Stacking)...")
meta_model.fit(X_stack, y_stack)

# Evaluate Stacking Score
stack_oof_preds = meta_model.predict(X_stack)
stack_oof_preds = np.clip(stack_oof_preds, 0, 100) # Clip to valid range

stack_rmse = np.sqrt(mean_squared_error(y_stack, stack_oof_preds))
print(f"\nStacked OOF RMSE: {stack_rmse:.6f}")
print(f"Best Alpha: {meta_model.alpha_}")

# The score is stored as negative RMSE (because sklearn maximizes scores)
honest_rmse = -1 * meta_model.best_score_
print(f"Honest Stacking CV RMSE: {honest_rmse:.5f}")

# Interpretability: Show the coefficients (Weights)
# This shows how much the Meta-Learner trusts each model.
coefs = pd.Series(meta_model.coef_, index=oof_df.columns)
print("\nMeta-Learner Weights:")
print(coefs.sort_values(ascending=False))

Preparing Stacking Data...
Stacking Models: ['cb', 'lgb', 'nn', 'ridge', 'xgb']
Training Meta-Learner (Ridge Stacking)...

Stacked OOF RMSE: 8.669918
Best Alpha: 500.0
Honest Stacking CV RMSE: 8.67000

Meta-Learner Weights:
xgb      0.622
lgb      0.268
cb       0.193
nn       0.013
ridge   -0.096
dtype: float64


## Final Ensemble & Submission

### Generating Stacking Predictions
Now that we have trained the **Ridge Meta-Learner** on the Out-of-Fold (OOF) predictions, we use it to generate the final exam score predictions for the Test Set.Important: We do not manually multiply weights (e.g., $0.3 \times M_1 + 0.7 \times M_2$). Instead, we treat the Test Set predictions from our base models (XGBoost, CatBoost, NN, etc.) as a **new feature matrix**. We feed this matrix into our trained `RidgeCV` model, which applies its learned coefficients (including any negative bias corrections) to output the final regression value.
  
The final prediction is effectively:

$$
\text{Final Prediction} = \alpha + \sum_{i=1}^{N} (\beta_i \times \text{Model}_i(X))
$$


Where $\beta_i$ are the learned coefficients (some positive, some potentially negative) that minimize the RMSE. This approach leverages the unique strengths of every model in the ensemble, ensuring that even "weaker" models contribute by correcting the errors of the stronger ones.

In [10]:
# Generate Final Submission
final_test_preds = meta_model.predict(X_test_stack)
final_test_preds = np.clip(final_test_preds, 0, 100)

submission_df[TARGET] = final_test_preds
submission_df.to_csv('submission.csv', index=False)
print("Saved: submission.csv")

print('SUBMISSION')
print('==========')

print(submission_df.head(10))

Saved: submission.csv
SUBMISSION
       id  exam_score
0  630000      70.324
1  630001      70.435
2  630002      88.235
3  630003      54.925
4  630004      47.137
5  630005      70.155
6  630006      73.203
7  630007      58.366
8  630008      78.512
9  630009      89.523
